### Plotting decision boundaries for machine learning algorithms in Python

In this notebook you will run code that allows you to visualise the decision boundaries of any classifier. We will use Logistic Regression. A popular diagnostic for understanding the decisions made by a classification algorithm is the decision surface. This is a plot that shows how a trained machine learning algorithm predicts a coarse grid across the input feature space. A decision surface plot is a powerful tool for understanding how a given model “sees” the prediction task and how it has decided to divide the input feature space by class label.

In [ ]:
# decision surface for logistic regression on a binary classification dataset
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs
from sklearn.metrics import accuracy_score

In [ ]:
# generate dataset
X, y = make_blobs(n_samples=1000, centers=2, n_features=2, random_state=1, cluster_std=3)
X_df = pd.DataFrame(X, columns=['x1','x2'])
X_df.head()


In [ ]:
y_df = pd.DataFrame(y, columns=["class"])
y_df.head()

In [ ]:
frames = [X_df, y_df]
data = pd.concat(frames, axis=1)
data.head()

In [ ]:
# create scatter plot for samples from each class
sns.scatterplot(x="x1", y="x2", hue='class', data=data)

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
# fit the model
model.fit(X, y)

Once the model is defined, we can use it to make a prediction for the training dataset to get an idea of how well it learned to divide the feature space of the training dataset and assign labels.

In [ ]:
# make predictions
yhat = model.predict(X)
# evaluate the predictions
acc = accuracy_score(y, yhat)
print('Accuracy: %.3f' % acc)

### Plot a decision surface

We can create a decision boundry by fitting a model on the training dataset, then using the model to make predictions for a grid of values across the input domain. Once we have the grid of predictions, we can plot the values and their class label.
A scatter plot could be used if a fine enough grid was taken. A better approach is to use a contour plot that can interpolate the colors between the points.
The contourf() Matplotlib function can be used.

This requires a few steps:
First, we need to define a grid of points across the feature space.
To do this, we can find the minimum and maximum values for each feature and expand the grid one step beyond that to ensure the whole feature space is covered.

In [ ]:
# define bounds of the domain
min1, max1 = X[:, 0].min()-1, X[:, 0].max()+1
min2, max2 = X[:, 1].min()-1, X[:, 1].max()+1

We can then create a uniform sample across each dimension using the arange() function at a chosen resolution. We will use a refinement_level of 0.1 in this case.

In [ ]:
# define the x and y scale
x1grid = np.arange(min1, max1, 0.1)
x2grid = np.arange(min2, max2, 0.1)


Now we need to turn this into a grid. We can use the meshgrid() NumPy function to create a grid from these two vectors.
If the first feature x1 is our x-axis of the feature space, then we need one row of x1 values of the grid for each point on the y-axis. Similarly, if we take x2 as our y-axis of the feature space, then we need one column of x2 values of the grid for each point on the x-axis. The meshgrid() function will do this for us, duplicating the rows and columns for us as needed. It returns two grids for the two input vectors. The first grid of x-values and the second of y-values, organized in an appropriately sized grid of rows and columns across the feature space.

In [ ]:
# create all of the lines and rows of the grid 
# each point in xx and yy reprsents a point in the 2d space
xx, yy = np.meshgrid(x1grid, x2grid)
xx.shape, yy.shape

Now to obtain all possible points we ravel xx and yy, i.e. we flatten them
into a simple vector using xx.ravel(), yy.ravel()
then we combine the two vectors into one matrix that have two column:
xx in the first column and yy in the second column

In [ ]:
xx.ravel().shape

In [ ]:
grid = np.c_[xx.ravel(), yy.ravel()]
grid.shape

In [ ]:
grid[:10]

We can then feed this into our model and get a prediction for each point in the grid.
Note that the points in 'grid' of the mehsgrid do not necessarily correspond to
particular data points in the dataset, they constitute hypothetical
data points that would be classified by the classifier in a specific way.

In [ ]:
# make predictions for the grid
yhat = model.predict(grid)

Now, we have a grid of values across the feature space and the class labels as predicted by our model. Next, we need to plot the grid of values as a contour plot.  The contourf() function takes separate grids for each axis, just like what was returned from our prior call to meshgrid(). So we can use xx and yy that we prepared earlier and simply reshape the predictions (yhat) from the model to have the same shape.

In [ ]:
# reshape the predictions back into a grid
zz = yhat.reshape(xx.shape)
zz.shape

We then plot the decision surface with a two-color colormap.

In [ ]:
# plot the grid of x, y and z values as a surface
plt.contourf(xx, yy, zz, cmap='Paired')

We can then plot the actual points of the dataset over the top to see how well they were separated by the logistic regression decision surface.

In [ ]:
plt.contourf(xx, yy, zz, cmap='Paired')
# create scatter plot for samples from each class
for class_value in range(2):
    # get row indexes for samples with this class
    row_ix = np.where(y == class_value)
    # create scatter of these samples
    plt.scatter(X[row_ix, 0], X[row_ix, 1], cmap='Paired')